In [2]:
from langchain_openrouter import ChatOpenRouter
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENROUTER_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENROUTER_API_KEY not found")

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

Bro API KEY Variable exists


In [3]:
# TASK - 1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")
])

In [5]:
# TASK - 2 [LLM]

llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

In [6]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [7]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text": text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

In [8]:
# TASK - 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

# TASK - 2 [LLM]
llm = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

# TASK - 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

In [9]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [10]:
def insta_chain(text: dict):
    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    # TASK - 2 [LLM]
    llm_insta = ChatOpenRouter(model="openai/gpt-4o-mini", temperature=0)

    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_insta | str_parser

    result = chain_insta.invoke({"text": text})

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

In [11]:
final_chain = (
    prompt_template |
    llm |
    str_parser |
    dictionary_maker_runnable |
    RunnableParallel(branches={"linkedin": chain_linkedin, "instagram": insta_chain_runnable})
)

In [12]:
final_chain.invoke({"input": "KGF"})

{'branches': {'linkedin': '🌟 **Unveiling the Power of Ambition: The Story of "KGF"** 🌟\n\n"KGF" (Kolar Gold Fields) is more than just an Indian action-drama film; it\'s a gripping tale of resilience, ambition, and the relentless pursuit of power. Set against the backdrop of the late 1970s and early 1980s, we follow the journey of Rocky, a young man from humble beginnings who dreams of wealth and influence.\n\nAs Rocky ventures into the Kolar Gold Fields in Karnataka, he finds himself navigating a treacherous world filled with rival gangs and the oppressive grip of the ruthless mining lord, Garuda. The film brilliantly captures his transformation from a street fighter to a formidable force within the gold mining empire, all while tackling themes of ambition and the struggle against oppression.\n\nWith its high-octane action sequences and dramatic storytelling, "KGF" has not only captivated audiences but also sparked conversations about socio-political issues that resonate even today. \n

In [13]:
# TASK - 1 [Beautify Function]

def beautify(final_response: dict) -> dict:
    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']
    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK - 2 [Beautified Chain]
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke({"input": "Pushpa"})

{'linkedin': '🌟 Exciting Film Spotlight: "Pushpa: The Rise" 🌟\n\nI recently had the chance to dive into the gripping world of "Pushpa: The Rise," an Indian action-drama film directed by Sukumar that took the cinematic landscape by storm in 2021. 🎬\n\nThe film follows the journey of Pushpa Raj, a laborer entangled in the red sandalwood smuggling trade in the lush forests of Andhra Pradesh. Portrayed brilliantly by Allu Arjun, Pushpa\'s ascent through the criminal underworld is a thrilling ride filled with ambition, betrayal, and the relentless pursuit of power. 💪\n\nWhat truly stands out is the film\'s exploration of personal relationships, particularly Pushpa\'s love interest, played by the talented Rashmika Mandanna. Their dynamic adds depth to the narrative, making it not just a tale of crime, but also of human connection. ❤️\n\nWith its strong performances, vibrant visuals, and a narrative that keeps you on the edge of your seat, "Pushpa: The Rise" is a testament to the power of sto